<a href="https://colab.research.google.com/github/mhage82/ARI510/blob/main/lab1_starter_mohammad_hassan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ARI 410/510 — Lab 1 Starter Notebook

*AI Disclosure: Claude Opus 4.8 was used to generate the initial version of this notebook.*

This notebook gets you off the ground: it **loads the data**, **cleans it**, makes a **train / dev / test split**, and runs **one $k$-NN model**.

**How to use this:** In Colab, go to *File → Save a copy in Drive* and work in your copy. Then run the cells top to bottom, read the comments, and build from the `# YOUR TURN` sections.

The choices made here (how to handle missing values, how to encode categories, the split sizes) are **reasonable starting points, not the only right answers** — you're encouraged to change them and say why in your report.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_score, recall_score, f1_score)

## 1. Load the data

Download `heart_disease_uci.csv` from Canvas (or Kaggle) and get it into this notebook.

- **In Colab:** run the cell below and pick the file from your computer, **or** upload it to the file panel on the left and skip the upload cell.
- **Locally (Jupyter):** just put the CSV next to this notebook.


In [ ]:
# --- Colab upload (skip if you added the file another way) ---
try:
    import google.colab
    from google.colab import files
    uploaded = files.upload()   # choose heart_disease_uci.csv
except ImportError:
    pass  # not on Colab; assume the file is alongside the notebook

In [ ]:
df = pd.read_csv('heart_disease_uci.csv')
print(df.shape)
df.head()

In [ ]:
# Quick look: column types and how many values are missing
df.info()
print('\nMissing values per column:')
print(df.isna().sum())

## 2. Define the label and the features

The raw target `num` runs from 0 to 4 (0 = no disease, 1–4 = increasing severity). For this lab we treat it as a **binary** problem: disease present (1) vs. absent (0). You could instead keep it multi-class — that's a design choice you can revisit.


In [ ]:
# Binary label: 1 if any disease is present, else 0
y = (df['num'] > 0).astype(int)
print(y.value_counts())

# Features: drop the id and the raw target. (You might also experiment with
# dropping 'dataset', which records the source hospital.)
X = df.drop(columns=['id', 'num'])

## 3. Clean and encode

Two simple, standard moves so the models will run:
- **One-hot encode** the categorical columns (turn text categories into 0/1 columns).
- **Fill missing values** with the column median.

These are deliberately basic. Better options exist (e.g., encoding inside a scikit-learn `ColumnTransformer`, or smarter imputation) — a good thing to improve on if you have time.


In [ ]:
X = pd.get_dummies(X, drop_first=True)          # categoricals -> 0/1 columns
X = X.fillna(X.median(numeric_only=True))        # simple missing-value handling
print('Feature matrix shape:', X.shape)
X.head()

## 4. Train / dev / test split

Three sets, not two:
- **train** — fit the models on this.
- **dev** — compare your configurations on this while you tune.
- **test** — *touch this only once*, at the very end, for your final numbers.

We use a 60 / 20 / 20 split, stratified so each set keeps the same class balance.


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=42)
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print(f'train: {len(X_train)}   dev: {len(X_dev)}   test: {len(X_test)}')

## 5. One example model: $k$-NN

$k$-NN compares a new point to its nearest neighbors, so the **scale** of each feature matters (a feature measured in the hundreds would otherwise dominate one measured 0–1). We put a `StandardScaler` in a `Pipeline` with the classifier so scaling is fit on the training data only.

We evaluate on the **dev** set — not the test set.


In [ ]:
knn = Pipeline([
    ('scale', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
])
knn.fit(X_train, y_train)

dev_pred = knn.predict(X_dev)
print('Dev-set performance for k=5:')
print(classification_report(y_dev, dev_pred, digits=3))
print('Confusion matrix (rows=true, cols=pred):')
print(confusion_matrix(y_dev, dev_pred))

## 6. YOUR TURN

Everything above is the scaffold. The actual lab is here.

**Run the comparison for your track:**
- **410:** run $k$-NN, logistic regression, and SVM across the specific hyperparameter settings in the assignment (at least 15 runs), evaluating each on the **dev** set. Then take the best of each and evaluate **once** on the **test** set.
- **510:** design your own comparison across at least five classifiers of your choice.

Useful imports:
```python
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
```
Every model can go in the same `Pipeline([('scale', StandardScaler()), ('model', ...)])` pattern used above.


In [ ]:
# YOUR TURN: loop over models and hyperparameter settings, record dev-set
# precision / recall / f1 / accuracy for each, and find the best per model.


### The data-value ablation (everyone)

Take your **single best configuration** and retrain it on 25%, 50%, and 100% of the *training* data (dev/test stay fixed). Plot performance vs. training-set size and comment on what it tells you about how much your result depends on the model versus the amount of data it saw.

The stub below shows the idea using the $k$-NN example — adapt it to your own best model.


In [ ]:
# YOUR TURN: adapt this to your BEST model, and pick the metric you care about.
fractions = [0.25, 0.50, 1.00]
sizes, scores = [], []
for frac in fractions:
    X_sub = X_train.sample(frac=frac, random_state=0)
    y_sub = y_train.loc[X_sub.index]
    model = Pipeline([('scale', StandardScaler()),
                      ('knn', KNeighborsClassifier(n_neighbors=5))]).fit(X_sub, y_sub)
    sizes.append(len(X_sub))
    scores.append(f1_score(y_dev, model.predict(X_dev)))

plt.plot(sizes, scores, marker='o')
plt.xlabel('Training examples used')
plt.ylabel('Dev F1')
plt.title('How much is the data worth?')
plt.show()
print(dict(zip(sizes, [round(s, 3) for s in scores])))


---
When you're happy with your experiments, write up your report (see the assignment PDF for the required sections and the Generative AI Use Statement). Questions → Discord `#homework-help`.
